In [50]:
import kagglehub

path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")

print("Path to dataset files:",path)

Using Colab cache for faster access to the 'twitter-entity-sentiment-analysis' dataset.
Path to dataset files: /kaggle/input/twitter-entity-sentiment-analysis


In [51]:
import os

print(os.listdir(path))

['twitter_validation.csv', 'twitter_training.csv']


In [52]:
import pandas as pd
import os

train_file = os.path.join(path, "twitter_training.csv")

df = pd.read_csv(train_file)

print(df.head())

   2401  Borderlands  Positive  \
0  2401  Borderlands  Positive   
1  2401  Borderlands  Positive   
2  2401  Borderlands  Positive   
3  2401  Borderlands  Positive   
4  2401  Borderlands  Positive   

  im getting on borderlands and i will murder you all ,  
0  I am coming to the borders and I will kill you...     
1  im getting on borderlands and i will kill you ...     
2  im coming on borderlands and i will murder you...     
3  im getting on borderlands 2 and i will murder ...     
4  im getting into borderlands and i can murder y...     


In [53]:
print(df.shape)

(74681, 4)


In [54]:
df = pd.read_csv(
    train_file,
    names=["Tweet_ID", "Entity", "Sentiment", "Tweet_Text"]
)

print(df.head())

   Tweet_ID       Entity Sentiment  \
0      2401  Borderlands  Positive   
1      2401  Borderlands  Positive   
2      2401  Borderlands  Positive   
3      2401  Borderlands  Positive   
4      2401  Borderlands  Positive   

                                          Tweet_Text  
0  im getting on borderlands and i will murder yo...  
1  I am coming to the borders and I will kill you...  
2  im getting on borderlands and i will kill you ...  
3  im coming on borderlands and i will murder you...  
4  im getting on borderlands 2 and i will murder ...  


In [55]:
df.to_csv("twitter_training_copy.csv", index=False)

print("Saved!")

Saved!


In [56]:
df.head()

,Tweet_ID,Entity,Sentiment,Tweet_Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [57]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [58]:
df = pd.read_csv("twitter_training_copy.csv")

print(df.head())

   Tweet_ID       Entity Sentiment  \
0      2401  Borderlands  Positive   
1      2401  Borderlands  Positive   
2      2401  Borderlands  Positive   
3      2401  Borderlands  Positive   
4      2401  Borderlands  Positive   

                                          Tweet_Text  
0  im getting on borderlands and i will murder yo...  
1  I am coming to the borders and I will kill you...  
2  im getting on borderlands and i will kill you ...  
3  im coming on borderlands and i will murder you...  
4  im getting on borderlands 2 and i will murder ...  


In [59]:
df = df[["Sentiment","Tweet_Text"]]

df.dropna(inplace=True)

print(df.head())

  Sentiment                                         Tweet_Text
0  Positive  im getting on borderlands and i will murder yo...
1  Positive  I am coming to the borders and I will kill you...
2  Positive  im getting on borderlands and i will kill you ...
3  Positive  im coming on borderlands and i will murder you...
4  Positive  im getting on borderlands 2 and i will murder ...


In [60]:
encoder = LabelEncoder()

df["Sentiment"] = encoder.fit_transform(
    df["Sentiment"]
)

In [61]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(
    df["Tweet_Text"]
)

In [62]:
X = tokenizer.texts_to_sequences(
    df["Tweet_Text"]
)

y = df["Sentiment"]

In [63]:
X = pad_sequences(
    X,
    maxlen=100
)

In [64]:
from tensorflow.keras.utils import to_categorical

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# One-hot encode the labels
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [65]:
model = Sequential([

    Embedding(
        input_dim=5000,
        output_dim=32
    ),

    SimpleRNN(32),

    Dense(
        4,
        activation="softmax"
    )
])

In [66]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy", # Changed to categorical_crossentropy
    metrics=["accuracy"]
)

In [67]:
model.fit(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test,y_test)
)

Epoch 1/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 51s 27ms/step - accuracy: 0.5652 - loss: 1.0379 - val_accuracy: 0.6828 - val_loss: 0.8089
Epoch 2/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 48s 26ms/step - accuracy: 0.7744 - loss: 0.6075 - val_accuracy: 0.7361 - val_loss: 0.6997
Epoch 3/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 46s 25ms/step - accuracy: 0.8484 - loss: 0.4188 - val_accuracy: 0.7618 - val_loss: 0.6596
Epoch 4/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 47s 26ms/step - accuracy: 0.8852 - loss: 0.3171 - val_accuracy: 0.7670 - val_loss: 0.6931
Epoch 5/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 80s 25ms/step - accuracy: 0.9088 - loss: 0.2538 - val_accuracy: 0.7527 - val_loss: 0.8191


In [68]:
loss,accuracy = model.evaluate(
    X_test,
    y_test
)

print("Accuracy:",accuracy)

463/463 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7527 - loss: 0.8191
Accuracy: 0.7527027130126953


In [69]:
predictions = model.predict(X_test)

y_pred = np.argmax(
    predictions,
    axis=1
)

# Convert y_test back to single integer labels for classification_report
y_test_labels = np.argmax(y_test, axis=1)

print(
    classification_report(
        y_test_labels,
        y_pred
    )
)

463/463 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step
              precision    recall  f1-score   support

           0       0.79      0.59      0.67      2696
           1       0.70      0.87      0.77      4380
           2       0.77      0.73      0.75      3605
           3       0.80      0.76      0.78      4119

    accuracy                           0.75     14800
   macro avg       0.76      0.74      0.74     14800
weighted avg       0.76      0.75      0.75     14800



In [74]:
sentence = input(
    "Enter a sentence: "
)

Enter a sentence: The service was excellent.


In [75]:
seq = tokenizer.texts_to_sequences(
    [sentence]
)

pad = pad_sequences(
    seq,
    maxlen=100
)

In [76]:
prediction = model.predict(
    pad
)

result = np.argmax(
    prediction
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


In [77]:
print(
    "Sentiment:",
    encoder.inverse_transform([result])[0]
)

Sentiment: Positive
